# Tabular Data Processing with Prior Labs MCP

In this recipe, we scrape hotel listings from the web, structure them into a table using an LLM, and then use a Haystack `Agent` equipped with [Prior Labs' TabPFN](https://priorlabs.ai/) tools to predict and fill in missing attributes.

**Services used:**
- [Firecrawl](https://www.firecrawl.dev/) — web scraping
- [Anthropic Claude](https://www.anthropic.com/) — LLM for extraction and agent reasoning
- [Prior Labs MCP](https://priorlabs.ai/) — tabular ML predictions via MCP tools

## Install dependencies

In [ ]:
!pip install -q haystack-ai anthropic-haystack mcp-haystack trafilatura firecrawl-haystack


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


## Set up API keys

You'll need three API keys:
- **Anthropic API key** — get one at [console.anthropic.com](https://console.anthropic.com/)
- **Firecrawl API key** — get one at [firecrawl.dev](https://www.firecrawl.dev/)
- **Prior Labs MCP token** — get one at [ux.priorlabs.ai](https://ux.priorlabs.ai/)

In [ ]:
import os
from getpass import getpass

if "ANTHROPIC_API_KEY" not in os.environ:
    os.environ["ANTHROPIC_API_KEY"] = getpass("Enter your Anthropic API key: ")
if "FIRECRAWL_API_KEY" not in os.environ:
    os.environ["FIRECRAWL_API_KEY"] = getpass("Enter your Firecrawl API key: ")
if "PRIOR_LABS_MCP_TOKEN" not in os.environ:
    os.environ["PRIOR_LABS_MCP_TOKEN"] = getpass("Enter your Prior Labs MCP token: ")

## Step 1: Scrape and structure hotel listings

We build a pipeline that fetches hotel search results with [`FirecrawlCrawler`](https://haystack.deepset.ai/integrations/firecrawl) and passes the raw text to Claude, which extracts the data into a structured Markdown table. Any missing fields are left blank - we'll fill those in with ML predictions in the next step.

In [ ]:
from haystack_integrations.components.generators.anthropic import AnthropicChatGenerator
from haystack_integrations.components.fetchers.firecrawl import FirecrawlCrawler
from haystack.components.builders import ChatPromptBuilder
from haystack.dataclasses import ChatMessage
from haystack import Pipeline


pipeline = Pipeline()
pipeline.add_component("fetcher", FirecrawlCrawler())
pipeline.add_component(
    "prompt_builder", 
    ChatPromptBuilder(
        template=[
            ChatMessage.from_system(
                "You are a hotel booking assistant. Extract the relevant information from the text and summarize it in a concise manner, as a Markdown table with the following columns: \n- Hotel name\n- Location\n- Distance from the target\n- Price per night\n- Room type\n- User rating\n- Location rating\n- Number of reviews\n- Number and types of beds\n- Amenities\n- Room size\n- Whether breakfast is included\n- Public transport availability\n\nIf any of the information is missing, leave the corresponding cell empty. Always return all the hotels from the input, even if some of the information is missing."
            ),
            ChatMessage.from_user(
                "{% for doc in documents %}{{ doc.content }}\n{% endfor %}"
            ),
        ],
        required_variables=["documents"],
    ),
)
pipeline.add_component(
    "summarizer", 
    AnthropicChatGenerator(
        model="claude-opus-4-6",
        generation_kwargs={
            "max_tokens": 10000,
        }
    ),
)

pipeline.connect("fetcher.documents", "prompt_builder.documents")
pipeline.connect("prompt_builder.prompt", "summarizer.messages")

🚅 Components
  - fetcher: FirecrawlCrawler
  - prompt_builder: ChatPromptBuilder
  - summarizer: AnthropicChatGenerator
🛤️ Connections
  - fetcher.documents -> prompt_builder.documents (list[Document])
  - prompt_builder.prompt -> summarizer.messages (list[ChatMessage])

In [ ]:
output = pipeline.run({
    "fetcher": {
        "urls": [
            "https://www.booking.com/searchresults.html?ss=Berlin%2C+Berlin+Federal+State%2C+Germany&efdco=1&label=gen173nr-10CAEoggI46AdIM1gEaLYBiAEBmAEzuAEHyAEM2AED6AEB-AEBiAIBqAIBuAKQgrTOBsACAdICJDllMDgyNTA1LTdlYmMtNDJkNi04NzFmLWUyYTRkNWM1NjM3ZtgCAeACAQ&aid=304142&lang=en-us&sb=1&src_elem=sb&src=index&dest_id=-1746443&dest_type=city&ac_position=0&ac_click_type=b&ac_langcode=xu&ac_suggestion_list_length=5&search_selected=true&search_pageview_id=0da2508885520c7d&ac_meta=GhAwZGEyNTA4ODg1NTIwYzdkIAAoATICeHU6BmJlcmxpbg%3D%3D&checkin=2026-05-05&checkout=2026-05-08&group_adults=1&no_rooms=1&group_children=0",
        ]
    }
})

In [ ]:
from IPython.display import Markdown, display

display(Markdown(output["summarizer"]["replies"][-1].text))

| Hotel Name | Location | Distance from Downtown | Price per Night | Room Type | User Rating | Location Rating | Number of Reviews | Beds | Amenities | Room Size | Breakfast Included | Public Transport |
|---|---|---|---|---|---|---|---|---|---|---|---|---|
| nestup - 80 qm Design Apartment in Kreuzberg | Friedrichshain-Kreuzberg, Berlin | 1.8 miles | $226 | Two-Bedroom Deluxe Apartment | 9.6 Exceptional | 10 | 14 | 3 beds (1 full, 2 queens) | Entire apartment, 2 bedrooms, 1 living room, 1 bathroom, 1 kitchen | 861 ft² | | |
| Hotel Berlin, Berlin (Radisson Individuals) | Mitte, Berlin | 1.3 miles | $188 | Cozy Small Room | 8.2 Very Good | | 19,671 | 1 queen bed | Sustainability certification | | | Subway Access |
| Lux 11 Berlin-Mitte | Mitte, Berlin | 1.5 miles | $169 | Superior Room | 8.7 Excellent | 9.5 | 2,074 | 1 queen bed | | | | Subway Access |
| Premier Inn Berlin City Süd | Neukölln, Berlin | 6.6 miles | $84 | Twin Room | 8.0 Very Good | | 1,737 | 2 twin beds | | | | |
| Hollywood Media Hotel am Kurfürstendamm | Charlottenburg-Wilmersdorf, Berlin | 2.5 miles | $137 | Comfort Double or Twin Room | 8.5 Very Good | 9.4 | 7,292 | 1 double or 2 twins | Sustainability certification | | | Subway Access |
| Premier Inn Berlin Airport | Treptow-Köpenick, Berlin | 10.8 miles | $77 | Standard Double Room | 8.6 Excellent | | 9,924 | 1 full bed | | | | |
| Steigenberger Hotel Am Kanzleramt | Mitte, Berlin | 0.6 miles | $219 | Superior Plus Room | 8.8 Excellent | 9.5 | 7,627 | 1 double or 2 twins | Sustainability certification, Free cancellation | | | Subway Access |
| Hotel Augusta Am Kurfürstendamm | Charlottenburg-Wilmersdorf, Berlin | 2.3 miles | $136 | Double or Twin Room | 8.1 Very Good | 9.4 | 2,961 | 1 double or 2 twins | | | | Subway Access |
| Living Hotel Berlin Mitte | Mitte, Berlin | 1.4 miles | $185 | Business Double Apartment | 8.6 Excellent | | 1,481 | 1 queen bed | Sustainability certification, Entire apartment, 1 bedroom, 1 bathroom | 248 ft² | | Subway Access |
| Townhouse Berlin | Charlottenburg-Wilmersdorf, Berlin | 2.3 miles | $155 | City Studio | 8.8 Excellent | 9.3 | 1,218 | 1 queen bed | Sustainability certification, Entire studio, 1 bathroom, Free cancellation | 280 ft² | | Subway Access |
| Numa Berlin Arc | Friedrichshain-Kreuzberg, Berlin | 1 mile | $136 | Double Room | 8.3 Very Good | | 5,268 | 1 queen bed | | | | Subway Access |
| Hotel Sachsenhof | Tempelhof-Schöneberg, Berlin | 1.6 miles | $144 | Triple Room | 8.5 Very Good | 9.3 | 5,260 | 2 beds (1 twin, 1 queen) | | | | Subway Access |
| Numa Berlin Torstrasse | Mitte, Berlin | 1.6 miles | $155 | Medium Room | 8.4 Very Good | 9.4 | 2,753 | 1 queen bed | | | | Subway Access |
| BENSIMON apartments Prenzlauer Berg | Pankow, Berlin | 2 miles | $176 | Studio Apartment with Garden View | 9.2 Wonderful | 9.5 | 595 | 2 beds (1 sofa bed, 1 queen) | Entire studio, 1 bathroom, 1 kitchen | 431 ft² | | Subway Access |
| Hotel MOA Berlin | Mitte, Berlin | 1.8 miles | $177 | Superior Double Room | 8.5 Very Good | | 5,554 | 1 queen bed | Sustainability certification | | | Subway Access |
| Industriepalast Berlin | Friedrichshain-Kreuzberg, Berlin | 3.1 miles | $25 | Single Bed in 8 Mixed Dormitory | 8.2 Very Good | | 5,961 | 1 bunk bed | Bed in dorm | | | Subway Access |
| 25hours Hotel Bikini Berlin | Charlottenburg-Wilmersdorf, Berlin | 1.8 miles | $246 | Medium Urban Twin | 8.7 Excellent | 9.5 | 2,169 | 2 twin beds | Sustainability certification | | | Subway Access |
| Telegraphenamt | Mitte, Berlin | 0.9 miles | $309 | Double Room | 8.8 Excellent | 9.7 | 727 | 1 king bed | Sustainability certification | | | |
| Myer's Hotel Berlin | Pankow, Berlin | 2 miles | $253 | Comfort Double or Twin Room | 9.0 Wonderful | | 993 | 1 double or 2 twins | | | Yes | Subway Access |
| Hotel Adlon Kempinski Berlin | Mitte, Berlin | 0.1 miles | $412 | Deluxe Double Room | 9.1 Wonderful | 9.8 | 3,904 | 5 beds (2 twins, 1 full, 1 king, 1 queen) | Free cancellation | | | Subway Access |
| THE GATE GARDEN Hotel | Mitte, Berlin | 1.1 miles | $173 | Double Room with Garden View | 8.8 Excellent | 9.3 | 722 | Multiple bed types | | | | Subway Access |
| Limehome Berlin Stresemannstr | Friedrichshain-Kreuzberg, Berlin | 1.1 miles | $122 | Double Room | 8.6 Excellent | | 1,463 | 1 queen bed | | | | Subway Access |
| Orania.Berlin | Friedrichshain-Kreuzberg, Berlin | 1.9 miles | $321 | Double Room (Orania.25) | 9.4 Wonderful | | 544 | 1 queen bed | | | | Subway Access |
| IntercityHotel Berlin Hauptbahnhof | Mitte, Berlin | 0.7 miles | $196 | Standard Double Room | 8.0 Very Good | 9.4 | 10,480 | 1 queen bed | Sustainability certification, Free cancellation | | | Subway Access |
| Holiday Inn Berlin City East Side | Friedrichshain-Kreuzberg, Berlin | 3 miles | $181 | Standard Twin Room with Courtyard View | 8.7 Excellent | 9.5 | 5,241 | 2 twin beds | | | | Subway Access |
| The Social Hub Berlin Alexanderplatz | Mitte, Berlin | 1.7 miles | $172 | Standard Queen Room | 8.6 Excellent | 9.3 | 9,300 | 1 queen bed | Free cancellation | | | Subway Access |
| Locke at East Side Gallery | Friedrichshain-Kreuzberg, Berlin | 2.6 miles | $130 | City Studio | 9.1 Wonderful | 9.3 | 3,133 | 1 queen bed | Entire studio, 1 bathroom | 237 ft² | | |
| InterContinental Berlin by IHG | Mitte, Berlin | 1.5 miles | $178 | Classic King Room | 8.8 Excellent | | 4,793 | 1 king bed | Sustainability certification | | | |

As you can see, some fields are missing for certain entries. Let's see how we can fill in that missing information using TabPFN models.

## Step 2: Connect to Prior Labs via MCP

[Prior Labs](https://priorlabs.ai/) exposes [TabPFN](https://github.com/PriorLabs/TabPFN), a pretrained tabular foundation model, through an MCP server. We load the toolset in the Prior Labs MCP using [MCPToolset](https://docs.haystack.deepset.ai/docs/mcptoolset), which gives the agent tools for uploading datasets and running fit/predict operations.

In [ ]:
from haystack_integrations.tools.mcp import MCPToolset, StreamableHttpServerInfo
from haystack.utils import Secret

# Get the server info from environment variables and create the toolset
# based on the tools exposed by the MCP server.
server_info = StreamableHttpServerInfo(
    url="https://api.priorlabs.ai/mcp/server",
    token=Secret.from_env_var("PRIOR_LABS_MCP_TOKEN"),
)
toolset = MCPToolset(server_info=server_info, eager_connect=True)

for tool in toolset.tools:
    print(f"{tool.name}: {tool.description}")

upload_dataset: Default entrypoint for file-based data.

Use this whenever the dataset is referenced as a file path, attachment, or URL.
Do not ask users to paste entire CSV contents into the conversation.

Initializes a direct upload to cloud storage. Returns a dataset_id and a
pre-signed upload_url valid for 60 minutes.

After calling this tool:
  1. HTTP PUT the file bytes directly to upload_url.
  2. Do NOT read the file contents into the conversation.
  3. Pass the returned dataset_id to fit_and_predict_from_dataset
     or predict_from_dataset.

Call this tool once per file — once for train.csv, once for test.csv.
The dataset_id has no implicit train/test meaning; label it yourself.

For ChatGPT/Claude web sessions without code execution:
  - still prefer this tool for real datasets
  - if upload cannot be executed in-session, do not paste full files inline;
    use an upload-capable MCP client/session.

Do NOT call this when the data is already a small inline array/table.

fit_a

## Step 3: Add an agent to fill in missing values

We extend the pipeline with a Haystack `Agent` that receives the structured hotel table and uses Prior Labs tools to predict missing attributes (location ratings, room sizes, transport options). Predicted values are marked in **bold** in the final output.

In [ ]:
from haystack.components.agents import Agent

llm = AnthropicChatGenerator(
    model="claude-opus-4-6",
    generation_kwargs={
        "max_tokens": 10000,
    }
)

pipeline.add_component(
    "final_prompt_builder",
    ChatPromptBuilder(
        # This particular provider expects the conversation
        # to end with a user message.
        template=[
            ChatMessage.from_user(
                "{% for reply in replies %}"
                "{{ reply.text }}\n"
                "{% endfor %}"
            ),
        ],
        required_variables=["replies"],
    ),
)
pipeline.add_component(
    "agent",
    Agent(
        chat_generator=llm,
        tools=toolset,
        system_prompt="You are a data scientist working with tabular data with a specialization in travel and hospitality. You have access to a set of tools that you should use to predict the missing attributes of hotels based on the information provided in the input table.\nBefore you use any of the tools, make sure to prepare the hotel data appropriately, like you would do for any tabular dataset, and interpret the results to create a human-friendly table with a structure similar to the input table.\nReturn only a Markdown-formatted table with the same columns as the input, but with the missing attributes filled in. Do not return any text other than the table. Mark filled attributes in bold. Do not return the tool outputs directly, or your reasoning, but only interpret them and return only the final table.",
    ),
)

pipeline.connect("summarizer.replies", "final_prompt_builder.replies")
pipeline.connect("final_prompt_builder.prompt", "agent.messages")

🚅 Components
  - fetcher: FirecrawlCrawler
  - prompt_builder: ChatPromptBuilder
  - summarizer: AnthropicChatGenerator
  - final_prompt_builder: ChatPromptBuilder
  - agent: Agent
🛤️ Connections
  - fetcher.documents -> prompt_builder.documents (list[Document])
  - prompt_builder.prompt -> summarizer.messages (list[ChatMessage])
  - summarizer.replies -> final_prompt_builder.replies (list[ChatMessage])
  - final_prompt_builder.prompt -> agent.messages (list[ChatMessage])

In [ ]:
output = pipeline.run({
    "fetcher": {
        "urls": [
            "https://www.booking.com/searchresults.html?ss=Berlin%2C+Berlin+Federal+State%2C+Germany&efdco=1&label=gen173nr-10CAEoggI46AdIM1gEaLYBiAEBmAEzuAEHyAEM2AED6AEB-AEBiAIBqAIBuAKQgrTOBsACAdICJDllMDgyNTA1LTdlYmMtNDJkNi04NzFmLWUyYTRkNWM1NjM3ZtgCAeACAQ&aid=304142&lang=en-us&sb=1&src_elem=sb&src=index&dest_id=-1746443&dest_type=city&ac_position=0&ac_click_type=b&ac_langcode=xu&ac_suggestion_list_length=5&search_selected=true&search_pageview_id=0da2508885520c7d&ac_meta=GhAwZGEyNTA4ODg1NTIwYzdkIAAoATICeHU6BmJlcmxpbg%3D%3D&checkin=2026-05-05&checkout=2026-05-08&group_adults=1&no_rooms=1&group_children=0",
        ]
    }
})

In [ ]:
# Do not display the entire response, but only extract
# the table after filling the gaps.
response = output["agent"]["last_message"].text
table = response[response.index("| Hotel Name |"):]
display(Markdown(table))

| Hotel Name | Location | Distance from Downtown | Price per Night | Room Type | User Rating | Location Rating | Number of Reviews | Beds | Amenities | Room Size | Breakfast Included | Public Transport |
|---|---|---|---|---|---|---|---|---|---|---|---|---|
| nestup - 80 qm Design Apartment in Kreuzberg | Friedrichshain-Kreuzberg, Berlin | 1.8 miles | $226 | Two-Bedroom Deluxe Apartment | 9.6 Exceptional | 10 | 14 | 3 beds (1 full, 2 queens) | Entire apartment, 2 bedrooms, 1 living room, 1 bathroom, 1 kitchen | 861 ft² | **No** | **Subway Access** |
| Hotel Berlin, Berlin (Radisson Individuals) | Mitte, Berlin | 1.3 miles | $188 | Cozy Small Room | 8.2 Very Good | **9.4** | 19,671 | 1 queen bed | Sustainability certification | **350 ft²** | **No** | Subway Access |
| Lux 11 Berlin-Mitte | Mitte, Berlin | 1.5 miles | $169 | Superior Room | 8.7 Excellent | 9.5 | 2,074 | 1 queen bed | **—** | **360 ft²** | **No** | Subway Access |
| Premier Inn Berlin City Süd | Neukölln, Berlin | 6.6 miles | $84 | Twin Room | 8.0 Very Good | **9.4** | 1,737 | 2 twin beds | **—** | **383 ft²** | **No** | **Subway Access** |
| Hollywood Media Hotel am Kurfürstendamm | Charlottenburg-Wilmersdorf, Berlin | 2.5 miles | $137 | Comfort Double or Twin Room | 8.5 Very Good | 9.4 | 7,292 | 1 double or 2 twins | Sustainability certification | **420 ft²** | **No** | Subway Access |
| Premier Inn Berlin Airport | Treptow-Köpenick, Berlin | 10.8 miles | $77 | Standard Double Room | 8.6 Excellent | **9.4** | 9,924 | 1 full bed | **—** | **321 ft²** | **No** | **Subway Access** |
| Steigenberger Hotel Am Kanzleramt | Mitte, Berlin | 0.6 miles | $219 | Superior Plus Room | 8.8 Excellent | 9.5 | 7,627 | 1 double or 2 twins | Sustainability certification, Free cancellation, No prepayment | **444 ft²** | **No** | Subway Access |
| Hotel Augusta Am Kurfürstendamm | Charlottenburg-Wilmersdorf, Berlin | 2.3 miles | $136 | Double or Twin Room | 8.1 Very Good | 9.4 | 2,961 | 1 double or 2 twins | — | **426 ft²** | **No** | Subway Access |
| Living Hotel Berlin Mitte | Mitte, Berlin | 1.4 miles | $185 | Business Double Apartment | 8.6 Excellent | **9.4** | 1,481 | 1 queen bed | Sustainability certification, Entire apartment, 1 bedroom, 1 bathroom | 248 ft² | **No** | Subway Access |
| Townhouse Berlin | Charlottenburg-Wilmersdorf, Berlin | 2.3 miles | $155 | City Studio | 8.8 Excellent | 9.3 | 1,218 | 1 queen bed | Sustainability certification, Free cancellation, Entire studio, 1 bathroom | 280 ft² | **No** | Subway Access |
| Numa Berlin Arc | Friedrichshain-Kreuzberg, Berlin | 1 miles | $136 | Double Room | 8.3 Very Good | **9.4** | 5,268 | 1 queen bed | **—** | **332 ft²** | **No** | Subway Access |
| Hotel Sachsenhof | Tempelhof-Schöneberg, Berlin | 1.6 miles | $144 | Triple Room | 8.5 Very Good | 9.3 | 5,260 | 2 beds (1 twin, 1 queen) | **—** | **433 ft²** | **No** | Subway Access |
| Numa Berlin Torstrasse | Mitte, Berlin | 1.6 miles | $155 | Medium Room | 8.4 Very Good | 9.4 | 2,753 | 1 queen bed | **—** | **324 ft²** | **No** | Subway Access |
| BENSIMON apartments Prenzlauer Berg | Pankow, Berlin | 2 miles | $176 | Studio Apartment with Garden View | 9.2 Wonderful | 9.5 | 595 | 2 beds (1 sofa bed, 1 queen) | Entire studio, 1 bathroom, 1 kitchen | 431 ft² | **No** | Subway Access |
| Hotel MOA Berlin | Mitte, Berlin | 1.8 miles | $177 | Superior Double Room | 8.5 Very Good | **9.4** | 5,554 | 1 queen bed | Sustainability certification | **310 ft²** | **No** | Subway Access |
| Industriepalast Berlin | Friedrichshain-Kreuzberg, Berlin | 3.1 miles | $25 | Single Bed in 8 Mixed Dormitory Room | 8.2 Very Good | **9.4** | 5,961 | 1 bunk bed | Bed in dorm | **361 ft²** | **No** | Subway Access |
| 25hours Hotel Bikini Berlin | Charlottenburg-Wilmersdorf, Berlin | 1.8 miles | $246 | Medium Urban Twin | 8.7 Excellent | 9.5 | 2,169 | 2 twin beds | Sustainability certification | **541 ft²** | **No** | Subway Access |
| Telegraphenamt | Mitte, Berlin | 0.9 miles | $309 | Double Room | 8.8 Excellent | 9.7 | 727 | 1 king bed | Sustainability certification | **631 ft²** | **No** | **Subway Access** |
| Myer's Hotel Berlin | Pankow, Berlin | 2 miles | $253 | Comfort Double or Twin Room | 9.0 Wonderful | **9.3** | 993 | 1 double or 2 twins | **—** | **521 ft²** | Yes | Subway Access |
| Hotel Adlon Kempinski Berlin | Mitte, Berlin | 0.1 miles | $412 | Deluxe Double Room | 9.1 Wonderful | 9.8 | 3,904 | 5 beds (2 twins, 1 full, 1 king, 1 queen) | Free cancellation, No prepayment | **588 ft²** | **No** | Subway Access |
| THE GATE GARDEN Hotel | Mitte, Berlin | 1.1 miles | $173 | Double Room with Garden View | 8.8 Excellent | 9.3 | 722 | Multiple bed types | **—** | **431 ft²** | **No** | Subway Access |
| Limehome Berlin Stresemannstr | Friedrichshain-Kreuzberg, Berlin | 1.1 miles | $122 | Double Room | 8.6 Excellent | **9.4** | 1,463 | 1 queen bed | **—** | **299 ft²** | **No** | Subway Access |
| Orania.Berlin | Friedrichshain-Kreuzberg, Berlin | 1.9 miles | $321 | Double Room (Orania.25) | 9.4 Wonderful | **9.6** | 544 | 1 queen bed | **—** | **443 ft²** | **No** | Subway Access |
| IntercityHotel Berlin Hauptbahnhof | Mitte, Berlin | 0.7 miles | $196 | Standard Double Room | 8.0 Very Good | 9.4 | 10,480 | 1 queen bed | Sustainability certification, Free cancellation, No prepayment | **355 ft²** | **No** | Subway Access |
| Holiday Inn Berlin City East Side | Friedrichshain-Kreuzberg, Berlin | 3 miles | $181 | Standard Twin Room with Courtyard View | 8.7 Excellent | 9.5 | 5,241 | 2 twin beds | **—** | **469 ft²** | **No** | Subway Access |
| The Social Hub Berlin Alexanderplatz | Mitte, Berlin | 1.7 miles | $172 | Standard Queen Room | 8.6 Excellent | 9.3 | 9,300 | 1 queen bed | Free cancellation | **299 ft²** | **No** | Subway Access |
| Locke at East Side Gallery | Friedrichshain-Kreuzberg, Berlin | 2.6 miles | $130 | City Studio | 9.1 Wonderful | 9.3 | 3,133 | 1 queen bed | Entire studio, 1 bathroom | 237 ft² | **No** | **Subway Access** |
| InterContinental Berlin by IHG | Mitte, Berlin | 1.5 miles | $178 | Classic King Room | 8.8 Excellent | **9.4** | 4,793 | 1 king bed | Sustainability certification | **388 ft²** | **No** | **Subway Access** |

## Conclusion

LLMs are remarkably good at turning messy, unstructured web content into clean, structured tables, but they struggle with numerical reasoning and will often leave gaps where data wasn't explicitly present on the page. This is where classical tabular ML still shines: given a few known examples, TabPFN can extrapolate missing values with no fine-tuning required.

The combination is powerful: use an LLM to handle the unstructured-to-structured step, then hand off to a tabular model for the gaps. Haystack's `Agent` with MCP tools makes it straightforward to wire these two worlds together in a single pipeline.